# DermaScreen: train the skin photo model (v2, clean)

**Before you start:** menu *Runtime > Change runtime type > T4 GPU*.
Then run the cells **one at a time, top to bottom** (Shift+Enter). Wait for each to finish.
If Colab ever restarts, just run from the top again: downloads are cached in Google Drive, so reruns are fast.

Stop after **step 8 (evaluation)** and check the accuracy before running the export cells.

Images stay in Colab/Drive. Do not upload them to GitHub, the app, or show them to users.
Fitzpatrick17k is CC BY-NC-SA 3.0. SCIN is under the SCIN Data Use License. Cite both in your paper.

## 1. Setup (mounts Google Drive, allow access when asked)

In [ ]:
import os, re, io, ast, json, time, random, subprocess, concurrent.futures as cf
import numpy as np, pandas as pd, requests
from PIL import Image

SEED = 42
random.seed(SEED); np.random.seed(SEED)

from google.colab import drive
drive.mount('/content/drive')
WORK   = '/content/drive/MyDrive/dermascreen/data'      # downloaded images are cached here
EXPORT = '/content/drive/MyDrive/dermascreen/export'    # model files go here
LOCAL  = '/content/data_local'                          # fast local copy used for training
os.makedirs(WORK, exist_ok=True); os.makedirs(EXPORT, exist_ok=True)

# ---- settings ----
MIN_IMAGES    = 70     # a condition with fewer images than this is not trained (stays rule-based)
MAX_PER_CLASS = 500    # at most this many images per condition
OTHER_MAX     = 600    # images of other conditions for the 'other' class
IMG           = 224

# condition code (must match skin_conditions.code in the database) -> words in dataset labels
PATTERNS = {
    'acne':             r'^acne',
    'eczema':           r'^eczema|atopic dermatitis',
    'tinea':            r'^tinea$|^tinea (corporis|pedis|cruris|manus|faciei)|ringworm',
    'scabies':          r'scabies',
    'impetigo':         r'impetigo',
    'melasma':          r'melasma',
    'psoriasis':        r'^psoriasis',
    'urticaria':        r'urticaria|hives',
    'vitiligo':         r'vitiligo',
    'folliculitis':     r'folliculitis',
    'molluscum':        r'molluscum',
    'warts':            r'wart|verruca',
    'tinea_versicolor': r'versicolor',
}
EXCLUDE = {'acne': r'keloidalis|excori|rosacea', 'tinea': r'versicolor'}

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True)

def map_label(label):
    l = str(label).lower().strip()
    for code, pat in PATTERNS.items():
        if re.search(pat, l) and not (code in EXCLUDE and re.search(EXCLUDE[code], l)):
            return code
    return None

def save_small(img_bytes, path, max_side=320):
    im = Image.open(io.BytesIO(img_bytes)).convert('RGB')
    im.thumbnail((max_side, max_side))
    os.makedirs(os.path.dirname(path), exist_ok=True)
    im.save(path, 'JPEG', quality=90)

def pick(df, code, n):
    d = df[df['code'] == code]
    return d.sample(min(n, len(d)), random_state=SEED)

rows = []   # every usable image: path, code, source, group
print('ready')

## 2. Dataset 1: Fitzpatrick17k
About 76% of its image links (the dermaamin.com site) are dead (HTTP 404). Only the atlasdermatologico.com.br images still work,
so we download only those, slowly and with retries.

In [ ]:
fz = pd.read_csv('https://raw.githubusercontent.com/mattgroh/fitzpatrick17k/main/fitzpatrick17k.csv')
fz['code']  = fz['label'].map(map_label)
fz['alive'] = fz['url'].astype(str).str.contains('atlasdermatologico')
print(fz.shape)
print(fz[fz['code'].notna()].groupby('code')['alive'].agg(total='size', working='sum').to_string())

In [ ]:
fz_live = fz[fz['alive']]
fz_sel = pd.concat([pick(fz_live, c, MAX_PER_CLASS) for c in PATTERNS] +
                   [fz_live[fz_live['code'].isna()].sample(OTHER_MAX // 2, random_state=SEED).assign(code='other')])
print(len(fz_sel), 'live images to get')

fail = {}
def get_fz(r):
    path = f"{WORK}/fitz/{r['code']}/{r['md5hash']}.jpg"
    item = {'path': path, 'code': r['code'], 'source': 'fitz', 'group': 'fz_' + r['md5hash']}
    if os.path.exists(path):
        return item
    for attempt in range(4):
        try:
            resp = requests.get(r['url'], timeout=30, headers={'User-Agent': 'Mozilla/5.0 (research download)'})
            if resp.status_code == 200 and len(resp.content) > 2000:
                save_small(resp.content, path)
                return item
            fail[resp.status_code] = fail.get(resp.status_code, 0) + 1
            if resp.status_code in (404, 410):
                return None
        except Exception as e:
            fail[type(e).__name__] = fail.get(type(e).__name__, 0) + 1
        time.sleep(1.5 * (attempt + 1))
    return None

with cf.ThreadPoolExecutor(4) as ex:       # few at once, so the site does not block us
    got = [g for g in ex.map(get_fz, [r for _, r in fz_sel.iterrows()]) if g]
rows += got
print('downloaded', len(got), 'of', len(fz_sel), '| failures:', fail)
print(pd.DataFrame(got)['code'].value_counts().to_string())

## 3. Dataset 2: Google SCIN (phone photos, closer to your real users)

In [ ]:
BUCKET = 'dx-scin-public-data'
os.makedirs(f'{WORK}/scin', exist_ok=True)
for f in ['scin_cases.csv', 'scin_labels.csv']:
    if not os.path.exists(f'{WORK}/scin/{f}'):
        r = sh(f'gsutil cp gs://{BUCKET}/dataset/{f} {WORK}/scin/{f}')
        print(f, r.returncode, r.stderr[:200])
cases  = pd.read_csv(f'{WORK}/scin/scin_cases.csv')
labels = pd.read_csv(f'{WORK}/scin/scin_labels.csv')
sc = cases.merge(labels, on='case_id')

def top_label(v):
    try:
        d = ast.literal_eval(v) if isinstance(v, str) else {}
        if not d: return None, 0.0
        k = max(d, key=d.get)
        return k, float(d[k])
    except Exception:
        return None, 0.0

tl = sc['weighted_skin_condition_label'].map(top_label)
sc['top_label'] = [t[0] for t in tl]
sc['top_w']     = [t[1] for t in tl]
sc['code']      = sc['top_label'].map(map_label)
sc_ok = sc[sc['top_w'] >= 0.35]
print(sc_ok['code'].value_counts(dropna=False).head(20).to_string())

In [ ]:
from google.cloud import storage
client = storage.Client.create_anonymous_client()
bucket = client.bucket(BUCKET)

img_cols = [c for c in sc.columns if re.fullmatch(r'image_\d+_path', c)]
parts = []
for k, cname in enumerate(img_cols):
    t = sc_ok[sc_ok[cname].notna()][['case_id', 'code', cname]].rename(columns={cname: 'gcs_path'})
    t['k'] = k
    parts.append(t)
sc_imgs = pd.concat(parts, ignore_index=True)

sc_sel = pd.concat([pick(sc_imgs, c, MAX_PER_CLASS) for c in PATTERNS] +
                   [sc_imgs[sc_imgs['code'].isna()].sample(min(OTHER_MAX // 2, int(sc_imgs['code'].isna().sum())), random_state=SEED).assign(code='other')])
print(len(sc_sel), 'images to get')

def get_sc(r):
    path = f"{WORK}/scin_img/{r['code']}/{r['case_id']}_{r['k']}.jpg"
    item = {'path': path, 'code': r['code'], 'source': 'scin', 'group': f"sc_{r['case_id']}"}
    if os.path.exists(path):
        return item
    try:
        save_small(bucket.blob(str(r['gcs_path'])).download_as_bytes(), path)
        return item
    except Exception:
        return None

with cf.ThreadPoolExecutor(16) as ex:
    got = [g for g in ex.map(get_sc, [r for _, r in sc_sel.iterrows()]) if g]
rows += got
print('downloaded', len(got), 'of', len(sc_sel))

## 4. Count what we really have, then pick the conditions to train

In [ ]:
df = pd.DataFrame(rows).drop_duplicates('path')
counts = df.groupby('code').size().sort_values(ascending=False)
print(counts.to_string())

keep = [c for c, n in counts.items() if n >= MIN_IMAGES]
print('\nTRAINING THESE:', keep)
print('TOO FEW IMAGES (rule-based only):', [c for c in counts.index if c not in keep])
if 'other' not in keep:
    raise SystemExit("The 'other' class has too few images, check the downloads above.")

df = df[df['code'].isin(keep)].copy()
df = df.sample(frac=1, random_state=SEED).groupby('code').head(MAX_PER_CLASS).reset_index(drop=True)
class_names = sorted(keep)
class_names.remove('other'); class_names.append('other')      # 'other' always last
idx = {c: i for i, c in enumerate(class_names)}
df['y'] = df['code'].map(idx)
print(class_names)

## 5. Copy the images to Colab's fast local disk (this prevents the 2-hour freeze)

In [ ]:
import shutil
for sub in ['fitz', 'scin_img']:
    if os.path.exists(f'{WORK}/{sub}'):
        shutil.copytree(f'{WORK}/{sub}', f'{LOCAL}/{sub}', dirs_exist_ok=True)
df['path'] = df['path'].str.replace(WORK, LOCAL, regex=False)
missing = [p for p in df['path'] if not os.path.exists(p)]
print('images:', len(df), '| missing on local disk:', len(missing))
assert not missing, 'some images are missing, rerun steps 2 and 3'

## 6. Split by case (no image of the same case in two sets)

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
sgkf = StratifiedGroupKFold(n_splits=6, shuffle=True, random_state=SEED)
fold = np.zeros(len(df), dtype=int)
for k, (_, te) in enumerate(sgkf.split(df, df['y'], df['group'])):
    fold[te] = k
df['split'] = np.where(fold == 0, 'test', np.where(fold == 1, 'val', 'train'))
print(df.groupby(['split', 'code']).size().unstack(0).to_string())

## 7. Build the model and train (about 15 to 25 minutes in total)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input
tf.random.set_seed(SEED)
print('GPU:', tf.config.list_physical_devices('GPU'))

BATCH = 32
augment = tf.keras.Sequential([
    layers.RandomFlip('horizontal_and_vertical'),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.2),
    layers.RandomBrightness(0.2),
])

def load(path, y):
    im = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    im = tf.image.resize(im, (IMG, IMG))
    return im, y

def make_ds(part, train):
    d = df[df['split'] == part]
    ds = tf.data.Dataset.from_tensor_slices((d['path'].values, d['y'].values.astype('int32')))
    ds = ds.map(load, num_parallel_calls=tf.data.AUTOTUNE)
    if train:
        ds = ds.shuffle(2000, seed=SEED)
        ds = ds.map(lambda x, y: (augment(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
    # MobileNetV2 wants pixels in [-1, 1]; the backend does the same (x/127.5 - 1)
    ds = ds.map(lambda x, y: (preprocess_input(x), y), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH).prefetch(tf.data.AUTOTUNE)

train_ds, val_ds, test_ds = make_ds('train', True), make_ds('val', False), make_ds('test', False)

base = MobileNetV2(input_shape=(IMG, IMG, 3), include_top=False, weights='imagenet')
base.trainable = False
inp = layers.Input((IMG, IMG, 3))
x = base(inp, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
out = layers.Dense(len(class_names), activation='softmax')(x)
model = models.Model(inp, out)

from sklearn.utils.class_weight import compute_class_weight
ytr = df[df['split'] == 'train']['y'].values
cw = compute_class_weight('balanced', classes=np.arange(len(class_names)), y=ytr)
class_weight = {i: float(w) for i, w in enumerate(cw)}
print(class_weight)

In [ ]:
# Stage 1: train only the new top layer (should take a few minutes)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
cb = [tf.keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True, monitor='val_accuracy')]
h1 = model.fit(train_ds, validation_data=val_ds, epochs=12, class_weight=class_weight, callbacks=cb)

In [ ]:
# Stage 2: fine-tune the top of MobileNetV2 slowly (batch-norm layers stay frozen)
base.trainable = True
for l in base.layers[:-60]:
    l.trainable = False
for l in base.layers:
    if isinstance(l, layers.BatchNormalization):
        l.trainable = False
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
cb = [tf.keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True, monitor='val_accuracy')]
h2 = model.fit(train_ds, validation_data=val_ds, epochs=15, class_weight=class_weight, callbacks=cb)
model.save(f'{EXPORT}/model_backup.keras')     # backup in Drive in case Colab disconnects
print('model backup saved')

## 8. Honest evaluation on images the model has never seen. STOP HERE and check the numbers.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

probs = model.predict(test_ds)
y_true = df[df['split'] == 'test']['y'].values
y_pred = probs.argmax(1)

acc  = float((y_true == y_pred).mean())
top3 = float(np.mean([t in np.argsort(p)[-3:] for t, p in zip(y_true, probs)]))
print(f'Test accuracy: {acc:.3f}   Top-3 accuracy: {top3:.3f}')
rep = classification_report(y_true, y_pred, labels=range(len(class_names)), target_names=class_names, output_dict=True, zero_division=0)
print(classification_report(y_true, y_pred, labels=range(len(class_names)), target_names=class_names, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=range(len(class_names)))
fig, ax = plt.subplots(figsize=(8, 7))
ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(class_names))); ax.set_xticklabels(class_names, rotation=90)
ax.set_yticks(range(len(class_names))); ax.set_yticklabels(class_names)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i, j], ha='center', va='center', color='black', fontsize=8)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual'); ax.set_title('Confusion matrix (test set)')
plt.tight_layout(); plt.savefig(f'{EXPORT}/confusion_matrix.png', dpi=150); plt.show()

In [ ]:
# How many test images are answered above each confidence line, and how often they are right
conf = probs.max(1)
for t in [0.3, 0.4, 0.5, 0.6, 0.7]:
    m = conf >= t
    print(f'threshold {t}: answers {m.mean()*100:5.1f}% of images, correct {(y_true[m]==y_pred[m]).mean()*100 if m.any() else 0:5.1f}% of those')

## 9. Export (run only if the accuracy in step 8 is acceptable)

In [ ]:
conv = tf.lite.TFLiteConverter.from_keras_model(model)
conv.optimizations = [tf.lite.Optimize.DEFAULT]
tfl = conv.convert()
open(f'{EXPORT}/dermascreen.tflite', 'wb').write(tfl)
json.dump(class_names, open(f'{EXPORT}/labels.json', 'w'))
json.dump({'test_accuracy': acc, 'top3_accuracy': top3, 'classes': class_names,
           'per_class_recall': {c: rep[c]['recall'] for c in class_names},
           'images': df.groupby('code').size().to_dict(),
           'sources': ['Fitzpatrick17k (CC BY-NC-SA 3.0)', 'Google SCIN (SCIN Data Use License)']},
          open(f'{EXPORT}/metrics.json', 'w'), indent=2)

# sanity check: the .tflite file must agree with the Keras model
it = tf.lite.Interpreter(model_content=tfl); it.allocate_tensors()
d_in, d_out = it.get_input_details()[0], it.get_output_details()[0]
xb, _ = next(iter(test_ds))
it.set_tensor(d_in['index'], xb[:1].numpy()); it.invoke()
print('keras :', np.round(model.predict(xb[:1], verbose=0)[0], 3))
print('tflite:', np.round(it.get_tensor(d_out['index'])[0], 3))
print('size  :', round(len(tfl) / 1e6, 1), 'MB')
print('classes:', class_names)
print('''
-- Run this in pgAdmin (as postgres) to record the new model:
UPDATE skin_classification_schema.model_versions SET is_active = FALSE WHERE is_active;
INSERT INTO skin_classification_schema.model_versions (name, version, training_data, accuracy, is_active, trained_at)
VALUES ('MobileNetV2', '1.0', 'Fitzpatrick17k + Google SCIN', %.4f, TRUE, CURRENT_DATE);
''' % acc)

In [ ]:
from google.colab import files
for f in ['dermascreen.tflite', 'labels.json', 'metrics.json', 'confusion_matrix.png']:
    files.download(f'{EXPORT}/{f}')

## After the download
1. Put `dermascreen.tflite` and `labels.json` in `backend/models/`.
2. Run the SQL printed above in pgAdmin.
3. In the backend terminal (venv active): `python -m pip install ai-edge-litert` (fallback: `python -m pip install tensorflow`), then restart uvicorn.
4. Submit a screening: the result should now use the photo model.
5. Keep `metrics.json` and the confusion matrix for your paper.